In [1]:
import pandas as pd
import os

RAW_DIR = '../data/raw'
orders = pd.read_csv(os.path.join(RAW_DIR, 'olist_orders_dataset.csv'))

date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

In [2]:
orders['has_incomplete_dates'] = (
    orders['order_status'].eq('delivered') &
    (orders['order_approved_at'].isnull() |
     orders['order_delivered_carrier_date'].isnull() |
     orders['order_delivered_customer_date'].isnull())
)

print(orders['has_incomplete_dates'].sum())

23


In [3]:
products = pd.read_csv('../data/raw/olist_products_dataset.csv')

incomplete_products = products[products['product_category_name'].isnull()]
display(incomplete_products.head(10))

# check how many of these products were actually ordered
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
sold_incomplete = order_items[order_items['product_id'].isin(incomplete_products['product_id'])]
print('Of the 610 incomplete products, how many appear in order_items (i.e. were actually sold):', sold_incomplete['product_id'].nunique())
print('Total order_item rows involving these products:', len(sold_incomplete))

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


Of the 610 incomplete products, how many appear in order_items (i.e. were actually sold): 610
Total order_item rows involving these products: 1603


In [4]:
sold_incomplete_value = order_items[order_items['product_id'].isin(incomplete_products['product_id'])]['price'].sum()
total_value = order_items['price'].sum()

print(f'Revenue from products missing category: R${sold_incomplete_value:,.2f}')
print(f'Total revenue (all order_items): R${total_value:,.2f}')
print(f'Percentage of total revenue: {sold_incomplete_value/total_value*100:.2f}%')


Revenue from products missing category: R$179,535.28
Total revenue (all order_items): R$13,591,643.70
Percentage of total revenue: 1.32%


In [5]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

print(products['product_category_name'].isnull().sum())  # should print 0

0


In [6]:
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

# show a few duplicate rows to see what they actually look like
dupes = geolocation[geolocation.duplicated(keep=False)]
display(dupes.sort_values('geolocation_zip_code_prefix').head(10))

# check: how many unique zip code prefixes exist vs total rows
print('Total rows:', len(geolocation))
print('Unique zip code prefixes:', geolocation['geolocation_zip_code_prefix'].nunique())

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
851,1001,-23.549825,-46.633970,sao paulo,SP
1062,1001,-23.550498,-46.634338,sao paulo,SP
1246,1001,-23.549292,-46.633559,sao paulo,SP
897,1001,-23.549292,-46.633559,sao paulo,SP
99,1001,-23.549292,-46.633559,sao paulo,SP
864,1001,-23.549825,-46.633970,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
818,1001,-23.551337,-46.634027,sao paulo,SP
1384,1001,-23.549292,-46.633559,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP


Total rows: 1000163
Unique zip code prefixes: 19015


In [7]:
# Step 1: remove exact duplicate rows (safe — these carry no extra information)
geolocation_dedup = geolocation.drop_duplicates()
print('Rows after removing exact duplicates:', len(geolocation_dedup))

# Step 2: collapse to one representative row per zip code prefix
geolocation_clean = geolocation_dedup.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print('Rows after collapsing to one per zip prefix:', len(geolocation_clean))

Rows after removing exact duplicates: 738332
Rows after collapsing to one per zip prefix: 19015


In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)

orders.to_csv('../data/processed/orders_clean.csv', index=False)
products.to_csv('../data/processed/products_clean.csv', index=False)
geolocation_clean.to_csv('../data/processed/geolocation_clean.csv', index=False)

# tables that needed no changes still get copied over, so processed/ has the full clean set
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

customers.to_csv('../data/processed/customers_clean.csv', index=False)
order_items.to_csv('../data/processed/order_items_clean.csv', index=False)
payments.to_csv('../data/processed/payments_clean.csv', index=False)
reviews.to_csv('../data/processed/reviews_clean.csv', index=False)
sellers.to_csv('../data/processed/sellers_clean.csv', index=False)
category_translation.to_csv('../data/processed/category_translation_clean.csv', index=False)

print(os.listdir('../data/processed'))

['.gitkeep', 'category_translation_clean.csv', 'customers_clean.csv', 'geolocation_clean.csv', 'orders_clean.csv', 'order_items_clean.csv', 'payments_clean.csv', 'products_clean.csv', 'reviews_clean.csv', 'sellers_clean.csv']


In [9]:
reviews = pd.read_csv('../data/processed/reviews_clean.csv')

dup_review_ids = reviews[reviews.duplicated(subset='review_id', keep=False)]
print('Total rows involved in duplicate review_ids:', len(dup_review_ids))
print('Number of distinct review_ids that are duplicated:', dup_review_ids['review_id'].nunique())

dup_review_ids.sort_values('review_id').head(10)

Total rows involved in duplicate review_ids: 1603
Number of distinct review_ids that are duplicated: 789


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
